
##### Read CSV File from Azure Data Lake Storage Account
 CSV Source File Path : "abfss://working-labs@datalakestorageaccountname.dfs.core.windows.net/bronze/daily-pricing/csv"

PARQUET Target  File Path : "abfss://working-labs@datalakestorageaccountname.dfs.core.windows.net/bronze/daily-pricing/parquet"
 
##### Spark Methods
- <a href="https://spark.apache.org/docs/latest/sql-getting-started.html#starting-point-sparksession" target="_blank">SparkSession</a>

- <a href="https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html" target="_blank">GenericDataFrameReader</a>: **`json`**,**`csv`**,  **`option (header,inferSchema)`** ,  **`schema`**

- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/column.html" target="_blank">PySparkSQLFunctions</a>: **`col`** , **`alias`** , **`cast`**


- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html" target="_blank">DataFrame</a> transformations and actions: 

| Method | Description |
| --- | --- |
| **`select`** | Returns a new DataFrame by computing given expression for each element |
| **`drop`** | Returns a new DataFrame with a column dropped |
| **`filter`**, **`where`** | Filters rows using the given condition |
| **`sort`**, **`orderBy`** | Returns a new DataFrame sorted by the given expressions |
| **`dropDuplicates`**, **`distinct`** | Returns a new DataFrame with duplicate rows removed |
| **`withColumnRenamed`** | Returns a new DataFrame with a column renamed |
| **`withColumn`** | Returns a new DataFrame by adding a column or replacing the existing column that has the same name |
| **`limit`** , **`take`** | Returns a new DataFrame by taking the first n rows |
| **`groupBy`** | Groups the DataFrame using the specified columns, so we can run aggregation on them |

- Other <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/dataframe.html" target="_blank">DataFrame</a> methods: **`printSchema`**, **`display`**, **`createOrReplaceTempView`**

- <a href="https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html" target="_blank">GenericDataFrameWriter</a>: **`parquet`**, **`csv`**, **`mode (overwrite,append)`** 

In [0]:
sourceCSVFilePath = 'abfss://working-labs@adlsudadatalakehousedev.dfs.core.windows.net/bronze/daily-pricing/csv'
targetPARQUETFilePath = 'abfss://working-labs@adlsudadatalakehousedev.dfs.core.windows.net/bronze/daily-pricing/parquet'

In [0]:
sourceCSVFileDF = (spark.
                   read.
                   load(sourceCSVFilePath , format = "csv" )
)

In [0]:
sourceCSVFileDF = (spark.
                   read.
                   load(sourceCSVFilePath , format = "csv"  , header = "true")
)

In [0]:
sourceCSVFileDF = (spark.
                   read.
                   load(sourceCSVFilePath , format = "csv"  , header = "true" , inferSchema = "true")
)

In [0]:
from pyspark.sql.functions import col
(sourceCSVFileDF.
 withColumn("ARRIVALS_IN_KILOGRAMS" , col("ARRIVAL_IN_TONNES") * 1000)
)

In [0]:
sourceCSVFileTransDF = (sourceCSVFileDF.
 withColumn("ARRIVALS_IN_KILOGRAMS" , col("ARRIVAL_IN_TONNES") * 1000)
 )

In [0]:
display(sourceCSVFileTransDF)

In [0]:
from pyspark.sql.functions import sum
(sourceCSVFileTransDF.
groupBy("STATE_NAME","PRODUCT_NAME").
agg(sum("ARRIVALS_IN_KILOGRAMS")).
        show())


In [0]:
(sourceCSVFileTransDF.
groupBy("STATE_NAME","PRODUCT_NAME").
agg(sum("ARRIVALS_IN_KILOGRAMS").alias("TOTAL_ARRIVALS_IN_KILOGRAMS")).
        show())

In [0]:
(sourceCSVFileTransDF.
groupBy("STATE_NAME","PRODUCT_NAME").
agg(sum("ARRIVALS_IN_KILOGRAMS").alias("TOTAL_ARRIVALS_IN_KILOGRAMS")).
orderBy("TOTAL_ARRIVALS_IN_KILOGRAMS").
        show())

In [0]:
from pyspark.sql.functions import desc
(sourceCSVFileTransDF.
groupBy("STATE_NAME","PRODUCT_NAME").
agg(sum("ARRIVALS_IN_KILOGRAMS").alias("TOTAL_ARRIVALS_IN_KILOGRAMS")).
orderBy(desc("TOTAL_ARRIVALS_IN_KILOGRAMS")).
        show())

In [0]:
(sourceCSVFileDF
 .write
  .save(targetPARQUETFilePath))

In [0]:
(spark
.read
.load(targetPARQUETFilePath)
.show()
)